# Trout Texture Feature Extraction

This notebook extracts handcrafted texture features from trout scale images for the corrected dataset.

Purpose:
- Create an interpretable feature table for professor-requested texture analysis.
- Support later comparisons against CNN/ImageNet/SimCLR embeddings.
- Build features for two downstream labels: `known_bad` and readable-only `age4` (`0+`, `1+`, `2+`, `3+`).

Run `trout_new_dataset_eda.ipynb` first so `eda_outputs/master_table_new.csv` exists.

## 1. Setup

The default paths match the HPC layout. Start with `RUN_SAMPLE=True`; after the first run succeeds, set it to `False` for the full dataset.

In [ ]:
from __future__ import annotations

import os
import math
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter
from tqdm.auto import tqdm

try:
    from skimage import exposure, feature, filters, measure, transform, util
    from skimage.color import rgb2gray
except ImportError as exc:
    raise ImportError(
        "This notebook needs scikit-image. In the active Jupyter kernel, run: "
        "pip install scikit-image"
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

ROOT_DIR = Path(os.environ.get("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.environ.get("TROUT_CODE_DIR", str(ROOT_DIR / "code_new")))
EDA_OUTPUT_DIR = CODE_DIR / "eda_outputs"
FEATURE_OUTPUT_DIR = CODE_DIR / "feature_outputs"
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = EDA_OUTPUT_DIR / "master_table_new.csv"
TEXTURE_FEATURE_CSV = FEATURE_OUTPUT_DIR / "texture_features.csv"
TEXTURE_MASTER_CSV = FEATURE_OUTPUT_DIR / "master_with_texture_features.csv"

SEED = 100
RUN_SAMPLE = True
SAMPLE_SIZE = 50
IMAGE_SIZE = 128
USE_GABOR = False
USE_HOG = False

np.random.seed(SEED)

print("ROOT_DIR:", ROOT_DIR)
print("CODE_DIR:", CODE_DIR)
print("MASTER_CSV exists:", MASTER_CSV.exists())
print("RUN_SAMPLE:", RUN_SAMPLE, "SAMPLE_SIZE:", SAMPLE_SIZE)
print("IMAGE_SIZE:", IMAGE_SIZE, "USE_GABOR:", USE_GABOR, "USE_HOG:", USE_HOG)

## 2. Load Master Table

This notebook uses the merged table from the EDA notebook. It keeps all rows, including unlabeled images, because texture features may also be useful for later semi-supervised or visualization work.

In [ ]:
if not MASTER_CSV.exists():
    raise FileNotFoundError(
        f"Missing {MASTER_CSV}. Run trout_new_dataset_eda.ipynb first."
    )

master_df = pd.read_csv(MASTER_CSV)

required_cols = {"path", "scale_id", "fish_key", "label", "known_bad", "age4", "length_mm", "weight_g"}
missing_cols = required_cols - set(master_df.columns)
if missing_cols:
    raise ValueError(f"master table is missing columns: {sorted(missing_cols)}")

master_df["path_exists"] = master_df["path"].map(lambda p: Path(str(p)).exists())
print("master_df:", master_df.shape)
print("paths missing:", int((~master_df["path_exists"]).sum()))
print("label counts:")
display(master_df["label"].value_counts(dropna=False).sort_index())
print("age4 counts:")
display(master_df["age4"].value_counts(dropna=False).sort_index())
display(master_df.head())

In [ ]:
work_df = master_df[master_df["path_exists"]].copy()

if RUN_SAMPLE:
    work_df = (
        work_df
        .sample(n=min(SAMPLE_SIZE, len(work_df)), random_state=SEED)
        .sort_values(["river", "point", "fish_id", "image_idx"])
        .reset_index(drop=True)
    )
else:
    work_df = work_df.reset_index(drop=True)

print("Rows selected for feature extraction:", len(work_df))
display(work_df[["scale_id", "fish_key", "label", "known_bad", "age4", "path"]].head())

## 3. Image Preprocessing

The features are extracted from grayscale, resized images. Optional contrast normalization is applied so acquisition differences do not dominate simple texture statistics.

In [ ]:
def load_gray_image(path: str | Path, image_size: int = IMAGE_SIZE) -> np.ndarray:
    img = Image.open(path).convert("L")
    img = ImageOps.autocontrast(img)
    img = img.filter(ImageFilter.SHARPEN)
    img = img.resize((image_size, image_size), resample=Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = exposure.equalize_adapthist(arr, clip_limit=0.03)
    return np.clip(arr, 0.0, 1.0).astype(np.float32)


def summarize_array(prefix: str, values: np.ndarray) -> dict:
    values = np.asarray(values, dtype=np.float64).ravel()
    if values.size == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_p25": np.nan,
            f"{prefix}_p50": np.nan,
            f"{prefix}_p75": np.nan,
            f"{prefix}_max": np.nan,
        }
    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_std": float(np.std(values)),
        f"{prefix}_min": float(np.min(values)),
        f"{prefix}_p25": float(np.percentile(values, 25)),
        f"{prefix}_p50": float(np.percentile(values, 50)),
        f"{prefix}_p75": float(np.percentile(values, 75)),
        f"{prefix}_max": float(np.max(values)),
    }

sample_img = load_gray_image(work_df.iloc[0]["path"])
print("sample image shape:", sample_img.shape, "range:", (float(sample_img.min()), float(sample_img.max())))

## 4. Texture Feature Functions

Default feature groups are intentionally fast enough for the full dataset:
- intensity histogram/statistics;
- Sobel/Canny edge and ridge-like structure summaries;
- GLCM/Haralick-style co-occurrence texture;
- LBP local binary pattern histogram.

Gabor and HOG summaries are included as optional functions, but disabled by default because they are much slower on the full dataset.

In [ ]:
def intensity_features(img: np.ndarray) -> dict:
    feats = summarize_array("intensity", img)
    hist, _ = np.histogram(img, bins=16, range=(0, 1), density=True)
    for i, v in enumerate(hist):
        feats[f"intensity_hist_{i:02d}"] = float(v)
    return feats


def edge_features(img: np.ndarray) -> dict:
    sobel = filters.sobel(img)
    canny = feature.canny(img, sigma=1.5)
    feats = summarize_array("sobel", sobel)
    feats["canny_edge_density"] = float(canny.mean())

    labeled = measure.label(canny)
    props = measure.regionprops(labeled)
    areas = np.array([p.area for p in props], dtype=float)
    feats.update(summarize_array("edge_component_area", areas))
    feats["edge_component_count"] = int(len(props))
    return feats


def glcm_features(img: np.ndarray) -> dict:
    img_u8 = util.img_as_ubyte(img)
    img_q = (img_u8 // 32).astype(np.uint8)
    distances = [1, 4]
    angles = [0, np.pi / 2]
    glcm = feature.graycomatrix(
        img_q,
        distances=distances,
        angles=angles,
        levels=8,
        symmetric=True,
        normed=True,
    )

    feats = {}
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        vals = feature.graycoprops(glcm, prop)
        feats.update(summarize_array(f"glcm_{prop}", vals))
    return feats


def lbp_features(img: np.ndarray) -> dict:
    radius = 2
    n_points = 8 * radius
    img_u8 = util.img_as_ubyte(img)
    lbp = feature.local_binary_pattern(img_u8, n_points, radius, method="uniform")
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins), density=True)
    return {f"lbp_uniform_r2_{i:02d}": float(v) for i, v in enumerate(hist)}


def gabor_features(img: np.ndarray) -> dict:
    feats = {}
    frequencies = [0.08, 0.16, 0.32]
    thetas = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
    for freq in frequencies:
        for theta in thetas:
            real, imag = filters.gabor(img, frequency=freq, theta=theta)
            magnitude = np.sqrt(real ** 2 + imag ** 2)
            key = f"gabor_f{freq:.2f}_t{int(round(theta * 180 / np.pi)):03d}"
            feats.update(summarize_array(key, magnitude))
    return feats


def hog_features(img: np.ndarray) -> dict:
    hog_vec = feature.hog(
        img,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )
    return summarize_array("hog", hog_vec)


def extract_texture_features(path: str | Path) -> dict:
    img = load_gray_image(path)
    feats = {}
    feats.update(intensity_features(img))
    feats.update(edge_features(img))
    feats.update(glcm_features(img))
    feats.update(lbp_features(img))
    if USE_GABOR:
        feats.update(gabor_features(img))
    if USE_HOG:
        feats.update(hog_features(img))
    return feats

features = extract_texture_features(work_df.iloc[0]["path"])
print("number of features:", len(features))
list(features.items())[:10]

## 5. Extract Features

This cell can take time for the full dataset. If it fails or is too slow, keep `RUN_SAMPLE=True` and inspect the sample first.

In [ ]:
rows = []
errors = []

for row in tqdm(work_df.itertuples(index=False), total=len(work_df), desc="texture features"):
    try:
        feats = extract_texture_features(row.path)
        feats.update({
            "scale_id": row.scale_id,
            "fish_key": row.fish_key,
            "path": row.path,
        })
        rows.append(feats)
    except Exception as exc:
        errors.append({
            "scale_id": getattr(row, "scale_id", None),
            "path": getattr(row, "path", None),
            "error": repr(exc),
        })

texture_df = pd.DataFrame(rows)
error_df = pd.DataFrame(errors)

id_cols = ["scale_id", "fish_key", "path"]
feature_cols = [c for c in texture_df.columns if c not in id_cols]
texture_df = texture_df[id_cols + feature_cols]

print("texture_df:", texture_df.shape)
print("errors:", len(error_df))
if len(error_df):
    display(error_df.head(20))
display(texture_df.head())

In [ ]:
suffix = "sample" if RUN_SAMPLE else "full"
texture_path = FEATURE_OUTPUT_DIR / f"texture_features_{suffix}.csv"
errors_path = FEATURE_OUTPUT_DIR / f"texture_feature_errors_{suffix}.csv"

texture_df.to_csv(texture_path, index=False)
error_df.to_csv(errors_path, index=False)

print("saved texture features:", texture_path)
print("saved errors:", errors_path)

## 6. Merge Features Back to Labels

This creates the table that later modeling notebooks should use.

In [ ]:
texture_feature_cols = [c for c in texture_df.columns if c not in ["fish_key", "path"]]
master_texture_df = master_df.merge(
    texture_df.drop(columns=["fish_key", "path"], errors="ignore"),
    on="scale_id",
    how="left",
)

feature_cols = [c for c in texture_df.columns if c not in ["scale_id", "fish_key", "path"]]
master_texture_df["has_texture_features"] = master_texture_df[feature_cols].notna().all(axis=1) if feature_cols else False

print("master_texture_df:", master_texture_df.shape)
print("rows with texture features:", int(master_texture_df["has_texture_features"].sum()))
display(master_texture_df[["scale_id", "fish_key", "label", "known_bad", "age4", "length_mm", "weight_g", "has_texture_features"]].head())

In [ ]:
master_texture_path = FEATURE_OUTPUT_DIR / f"master_with_texture_features_{suffix}.csv"
master_texture_df.to_csv(master_texture_path, index=False)
print("saved merged table:", master_texture_path)

## 7. Quick Feature Sanity Checks

These summaries are not final model results. They only check whether extracted features vary by label enough to justify modeling.

In [ ]:
readable_texture = master_texture_df[master_texture_df["age4"].notna() & master_texture_df["has_texture_features"]].copy()
bad_texture = master_texture_df[master_texture_df["known_bad"].notna() & master_texture_df["has_texture_features"]].copy()

print("readable rows with age4:", readable_texture.shape)
print("rows with known_bad:", bad_texture.shape)

candidate_features = [
    "glcm_contrast_mean",
    "glcm_homogeneity_mean",
    "glcm_energy_mean",
    "glcm_correlation_mean",
    "canny_edge_density",
    "sobel_mean",
    "hog_mean",
    "hog_std",
]
candidate_features = [c for c in candidate_features if c in master_texture_df.columns]

if candidate_features and len(readable_texture):
    display(readable_texture.groupby("age4")[candidate_features].mean().round(4))
if candidate_features and len(bad_texture):
    display(bad_texture.groupby("known_bad")[candidate_features].mean().round(4))

## 8. Optional: Fast Texture-Only Baseline

This is a lightweight check only. Use fish-level splitting in the real modeling notebook to avoid leakage across multiple images from the same fish.

In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, balanced_accuracy_score
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
except ImportError:
    print("Skipping optional baseline. Install scikit-learn to run this cell: pip install scikit-learn")
else:
    model_df = master_texture_df[master_texture_df["age4"].notna() & master_texture_df["has_texture_features"]].copy()
    feature_cols = [c for c in texture_df.columns if c not in ["scale_id", "fish_key", "path"]]
    if len(model_df) >= 50 and len(model_df["fish_key"].unique()) >= 10:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
        train_idx, test_idx = next(splitter.split(model_df, model_df["age4"], groups=model_df["fish_key"]))
        train_df = model_df.iloc[train_idx]
        test_df = model_df.iloc[test_idx]

        clf = make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=300,
                random_state=SEED,
                class_weight="balanced_subsample",
                n_jobs=-1,
            ),
        )
        clf.fit(train_df[feature_cols], train_df["age4"].astype(int))
        pred = clf.predict(test_df[feature_cols])
        print("Fish overlap:", len(set(train_df["fish_key"]) & set(test_df["fish_key"])))
        print("Balanced accuracy:", balanced_accuracy_score(test_df["age4"].astype(int), pred))
        print(classification_report(test_df["age4"].astype(int), pred, target_names=["0+", "1+", "2+", "3+"], digits=4))
    else:
        print("Not enough labeled texture rows for the optional baseline.")

## 9. Next Steps

After the full feature table is created:
- compare `texture-only` against `length/weight-only`;
- compare `CNN/ImageNet embedding-only` against `CNN + texture + length/weight`;
- keep fish-level splits for every reported metric;
- focus the main age task on readable images with `age4` labels.